In [2]:
import sys
print(sys.executable)

/Users/christianormsby/Documents/premier-league-performance-intelligence/.venv/bin/python3.13


In [3]:
import pandas as pd

In [19]:
df = pd.read_csv("../data/raw/E0.csv")

In [20]:
df.shape

(380, 132)

In [6]:
df.columns.tolist()

['Div',
 'Date',
 'Time',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR',
 'AR',
 'B365H',
 'B365D',
 'B365A',
 'BFDH',
 'BFDD',
 'BFDA',
 'BMGMH',
 'BMGMD',
 'BMGMA',
 'BVH',
 'BVD',
 'BVA',
 'BWH',
 'BWD',
 'BWA',
 'CLH',
 'CLD',
 'CLA',
 'LBH',
 'LBD',
 'LBA',
 'PSH',
 'PSD',
 'PSA',
 'MaxH',
 'MaxD',
 'MaxA',
 'AvgH',
 'AvgD',
 'AvgA',
 'BFEH',
 'BFED',
 'BFEA',
 'B365>2.5',
 'B365<2.5',
 'P>2.5',
 'P<2.5',
 'Max>2.5',
 'Max<2.5',
 'Avg>2.5',
 'Avg<2.5',
 'BFE>2.5',
 'BFE<2.5',
 'AHh',
 'B365AHH',
 'B365AHA',
 'PAHH',
 'PAHA',
 'MaxAHH',
 'MaxAHA',
 'AvgAHH',
 'AvgAHA',
 'BFEAHH',
 'BFEAHA',
 'B365CH',
 'B365CD',
 'B365CA',
 'BFDCH',
 'BFDCD',
 'BFDCA',
 'BMGMCH',
 'BMGMCD',
 'BMGMCA',
 'BVCH',
 'BVCD',
 'BVCA',
 'BWCH',
 'BWCD',
 'BWCA',
 'CLCH',
 'CLCD',
 'CLCA',
 'LBCH',
 'LBCD',
 'LBCA',
 'PSCH',
 'PSCD',
 'PSCA',
 'MaxCH',
 'MaxCD',
 'MaxCA',
 'AvgCH',
 'Avg

In [21]:
home = df[[
    'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR'
]].copy()

home.columns = [
    'Date', 'Team', 'Opponent',
    'Goals', 'GoalsConceded', 'FTR'
]

away = df[[
    'Date', 'AwayTeam', 'HomeTeam',
    'FTAG', 'FTHG', 'FTR'
]].copy()

away.columns = [
    'Date', 'Team', 'Opponent',
    'Goals', 'GoalsConceded', 'FTR'
]

print(home.shape)
print(away.shape)

(380, 6)
(380, 6)


In [22]:
team_matches = pd.concat(
    [home, away],
    ignore_index=True
)

team_matches.shape

(760, 6)

In [7]:
df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


In [24]:
def get_result(row):
    if row['Goals'] > row['GoalsConceded']:
        return 'W'
    elif row['Goals'] < row['GoalsConceded']:
        return 'L'
    else:
        return 'D'

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Columns: 132 entries, Div to BFECAHA
dtypes: float64(108), int64(16), str(8)
memory usage: 392.0 KB


In [25]:
team_matches['Result'] = team_matches.apply(
    get_result,
    axis=1
)

team_matches['Result'].value_counts()

Result
W    276
L    276
D    208
Name: count, dtype: int64

In [9]:
df.dtypes

Div             str
Date            str
Time            str
HomeTeam        str
AwayTeam        str
             ...   
MaxCAHA     float64
AvgCAHH     float64
AvgCAHA     float64
BFECAHH     float64
BFECAHA     float64
Length: 132, dtype: object

In [27]:
team_matches[
    team_matches['Team'].isin(['Man City', 'Man United'])
].groupby(['Team', 'Result']).size()

Team        Result
Man City    D          9
            L          6
            W         23
Man United  D         11
            L          7
            W         20
dtype: int64

In [28]:
team_matches['Points'] = team_matches['Result'].map({
    'W': 3,
    'D': 1,
    'L': 0
})

team_matches['GoalDifference'] = (
    team_matches['Goals'] - team_matches['GoalsConceded']
)

league_table_python = team_matches.groupby('Team').agg(
    Played=('Team', 'count'),
    Wins=('Result', lambda x: (x == 'W').sum()),
    Draws=('Result', lambda x: (x == 'D').sum()),
    Losses=('Result', lambda x: (x == 'L').sum()),
    GoalsFor=('Goals', 'sum'),
    GoalsAgainst=('GoalsConceded', 'sum'),
    GoalDifference=('GoalDifference', 'sum'),
    Points=('Points', 'sum')
).reset_index()

league_table_python = league_table_python.sort_values(
    ['Points', 'GoalDifference'],
    ascending=[False, False]
).reset_index(drop=True)

league_table_python['Position'] = (
    league_table_python.index + 1
)

league_table_python

,Team,Played,Wins,Draws,Losses,GoalsFor,GoalsAgainst,GoalDifference,Points,Position
0,Arsenal,38,26,7,5,71,27,44,85,1
1,Man City,38,23,9,6,77,35,42,78,2
2,Man United,38,20,11,7,69,50,19,71,3
3,Aston Villa,38,19,8,11,56,49,7,65,4
4,Liverpool,38,17,9,12,63,53,10,60,5
5,Bournemouth,38,13,18,7,58,54,4,57,6
6,Sunderland,38,14,12,12,42,48,-6,54,7
7,Brighton,38,14,11,13,52,46,6,53,8
8,Brentford,38,14,11,13,55,52,3,53,9
9,Chelsea,38,14,10,14,58,52,6,52,10


In [29]:
team_matches.to_csv(
    "../data/processed/team_matches.csv",
    index=False
)

print("Saved corrected team_matches.csv")
print(team_matches.shape)

Saved corrected team_matches.csv
(760, 9)


In [30]:
league_table_python.to_csv(
    "../data/processed/league_table.csv",
    index=False
)

print("Saved corrected league_table.csv")
print(league_table_python.shape)

Saved corrected league_table.csv
(20, 10)


In [10]:
df.select_dtypes(include="string").columns.tolist()

['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTR', 'HTR', 'Referee']

In [11]:
df.isna().sum()

Div          0
Date         0
Time         0
HomeTeam     0
AwayTeam     0
            ..
MaxCAHA      0
AvgCAHH      0
AvgCAHA      0
BFECAHH     22
BFECAHA     22
Length: 132, dtype: int64

In [12]:
df.isna().sum()[df.isna().sum() > 0]

BFDH          1
BFDD          1
BFDA          1
BMGMH         2
BMGMD         2
BMGMA         2
BVH           2
BVD           2
BVA           2
CLH          98
CLD          98
CLA          98
LBH          94
LBD          94
LBA          94
PSH         170
PSD         170
PSA         170
BFEH         20
BFED         20
BFEA         20
P>2.5       170
P<2.5       170
BFE>2.5      20
BFE<2.5      20
AHh           1
PAHH        170
PAHA        170
BFEAHH       20
BFEAHA       20
BFDCH         8
BFDCD         8
BFDCA         8
BVCH          8
BVCD          8
BVCA          8
CLCH        120
CLCD        120
CLCA        120
LBCH         99
LBCD         99
LBCA         99
PSCH        170
PSCD        170
PSCA        170
BFECH        22
BFECD        22
BFECA        22
PC>2.5      170
PC<2.5      170
BFEC>2.5     22
BFEC<2.5     22
PCAHH       170
PCAHA       170
BFECAHH      22
BFECAHA      22
dtype: int64

In [13]:
df.describe()

,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HF,AF,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
count,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,...,380.000000,380.000000,210.000000,210.000000,380.000000,380.000000,380.000000,380.000000,358.000000,358.000000
mean,1.526316,1.223684,0.692105,0.497368,13.839474,11.157895,4.502632,3.871053,10.652632,10.989474,...,1.919368,1.930447,1.964762,1.966143,1.942947,1.956474,1.873974,1.886105,1.990782,1.998883
std,1.169914,1.084779,0.777277,0.742593,4.949137,4.573217,2.244311,2.112085,3.215295,3.397173,...,0.096736,0.095349,0.110116,0.111749,0.092333,0.090155,0.083938,0.084617,0.096875,0.097022
min,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,0.000000,0.000000,3.000000,3.000000,...,1.680000,1.650000,1.680000,1.680000,1.750000,1.750000,1.710000,1.700000,1.810000,1.740000
25%,1.000000,0.000000,0.000000,0.000000,11.000000,8.000000,3.000000,2.000000,8.000000,9.000000,...,1.850000,1.850000,1.880000,1.880000,1.870000,1.880000,1.810000,1.820000,1.910000,1.920000
50%,1.000000,1.000000,1.000000,0.000000,14.000000,11.000000,4.000000,4.000000,10.000000,11.000000,...,1.930000,1.930000,1.970000,1.950000,1.930000,1.950000,1.865000,1.890000,1.980000,2.000000
75%,2.000000,2.000000,1.000000,1.000000,16.250000,14.000000,6.000000,5.000000,13.000000,13.000000,...,2.000000,2.000000,2.040000,2.050000,2.030000,2.030000,1.940000,1.950000,2.060000,2.077500
max,5.000000,5.000000,3.000000,4.000000,35.000000,30.000000,11.000000,11.000000,20.000000,21.000000,...,2.200000,2.150000,2.320000,2.330000,2.350000,2.280000,2.100000,2.080000,2.320000,2.220000


In [14]:
football_cols = [
    'FTHG', 'FTAG',
    'HTHG', 'HTAG',
    'HS', 'AS',
    'HST', 'AST',
    'HF', 'AF',
    'HC', 'AC',
    'HY', 'AY',
    'HR', 'AR'
]

df[football_cols].describe()

,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
count,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000
mean,1.526316,1.223684,0.692105,0.497368,13.839474,11.157895,4.502632,3.871053,10.652632,10.989474,5.394737,4.602632,1.665789,2.081579,0.050000,0.052632
std,1.169914,1.084779,0.777277,0.742593,4.949137,4.573217,2.244311,2.112085,3.215295,3.397173,2.729101,2.748469,1.229105,1.318101,0.241204,0.223591
min,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,0.000000,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,11.000000,8.000000,3.000000,2.000000,8.000000,9.000000,3.000000,3.000000,1.000000,1.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,0.000000,14.000000,11.000000,4.000000,4.000000,10.000000,11.000000,5.000000,4.000000,2.000000,2.000000,0.000000,0.000000
75%,2.000000,2.000000,1.000000,1.000000,16.250000,14.000000,6.000000,5.000000,13.000000,13.000000,7.000000,6.000000,2.000000,3.000000,0.000000,0.000000
max,5.000000,5.000000,3.000000,4.000000,35.000000,30.000000,11.000000,11.000000,20.000000,21.000000,14.000000,15.000000,5.000000,6.000000,2.000000,1.000000


In [15]:
df['FTR'].value_counts()

FTR
H    162
A    114
D    104
Name: count, dtype: int64

In [16]:
.value_counts()

SyntaxError: invalid syntax (747460836.py, line 1)

In [ ]:
df[['FTHG', 'FTAG']].mean()

In [ ]:
df[football_cols].mean()

In [ ]:
df.iloc[0]

In [ ]:
df[
    [
        'HomeTeam', 'AwayTeam',
        'FTHG', 'FTAG',
        'HS', 'AS',
        'HST', 'AST',
        'HC', 'AC',
        'HY', 'AY',
        'HR', 'AR',
        'FTR'
    ]
].head()

In [ ]:
home = df[
    [
        'HomeTeam', 'AwayTeam',
        'FTHG', 'FTAG',
        'HS', 'AS',
        'HST', 'AST',
        'HC', 'AC',
        'HY', 'HR',
        'FTR'
    ]
].copy()

In [ ]:
home = home.rename(
    columns={
        'HomeTeam': 'Team',
        'AwayTeam': 'Opponent',
        'FTHG': 'Goals',
        'FTAG': 'GoalsConceded',
        'HS': 'Shots',
        'AS': 'ShotsConceded',
        'HST': 'ShotsOnTarget',
        'AST': 'ShotsOnTargetConceded',
        'HC': 'Corners',
        'AC': 'CornersConceded',
        'HY': 'YellowCards',
        'HR': 'RedCards'
    }
)

In [ ]:
home['Venue'] = 'Home'

In [ ]:
home.head()

In [ ]:
away = df[
    [
        'AwayTeam', 'HomeTeam',
        'FTAG', 'FTHG',
        'AS', 'HS',
        'AST', 'HST',
        'AC', 'HC',
        'AY', 'AR',
        'FTR'
    ]
].copy()

In [ ]:
away.head()

In [ ]:
away = away.rename(
    columns={
        'AwayTeam': 'Team',
        'HomeTeam': 'Opponent',
        'FTAG': 'Goals',
        'FTHG': 'GoalsConceded',
        'AS': 'Shots',
        'HS': 'ShotsConceded',
        'AST': 'ShotsOnTarget',
        'HST': 'ShotsOnTargetConceded',
        'AC': 'Corners',
        'HC': 'CornersConceded',
        'AY': 'YellowCards',
        'AR': 'RedCards'
    }
)

In [ ]:
away['Venue'] = 'Away'

In [ ]:
away.head()

In [ ]:
team_matches = pd.concat([home, away], ignore_index=True)

In [ ]:
team_matches.shape

In [ ]:
def get_result(row):
    if row['FTR'] == 'D':
        return 'D'
    elif row['FTR'] == 'H' and row['Venue'] == 'Home':
        return 'W'
    elif row['FTR'] == 'A' and row['Venue'] == 'Away':
        return 'W'
    else:
        return 'L'

In [ ]:
team_matches['Result'] = team_matches.apply(get_result, axis=1)

In [ ]:
team_matches[['Team', 'Opponent', 'Venue', 'FTR', 'Result']].head(10)

In [ ]:
team_matches[team_matches['Venue'] == 'Away'][['Team', 'Opponent', 'Venue', 'FTR', 'Result']].head()

In [ ]:
team_matches[team_matches['Venue'] == 'Away'][['Team', 'Opponent', 'Venue', 'FTR', 'Result']].head(1)

In [ ]:
team_matches['Result'].value_counts()

In [ ]:
276+276+208

In [ ]:
team_matches['Team'].value_counts().sort_index()

In [ ]:
team_matches['Points'] = team_matches['Result'].map({
    'W': 3,
    'D': 1,
    'L': 0
})

In [ ]:
team_matches[['Team', 'Result', 'Points']].head(10)

In [ ]:
team_matches = team_matches.drop(columns=['FTR'])

In [ ]:
team_matches.head()

In [ ]:
team_matches['GoalDifference'] = (
    team_matches['Goals'] - team_matches['GoalsConceded']
)

In [ ]:
team_matches[['Team', 'Goals', 'GoalsConceded', 'GoalDifference']].head(10)

In [ ]:
(team_matches['Goals'] - team_matches['GoalsConceded'] == team_matches['GoalDifference']).all()

In [ ]:
team_matches.groupby('Team').size()

In [ ]:
team_matches['Result'].eq('W').sum()

In [ ]:
team_matches['Win'] = team_matches['Result'].eq('W')

In [ ]:
team_matches['Win'].head()

In [ ]:
team_matches.groupby('Team')['Win'].sum()

In [ ]:
team_matches.groupby('Team')['Win'].sum().sort_values(ascending=False)

In [ ]:
team_matches['Draw'] = team_matches['Result'].eq('D')

In [ ]:
team_matches['Draw'].head()

In [ ]:
team_matches.groupby('Team')['Draw'].sum().sort_values(ascending=False)

In [ ]:
team_matches['Loss'] = team_matches['Result'].eq('L')

In [ ]:
team_matches[['Team', 'Result', 'Win', 'Draw', 'Loss']].head(10)

In [ ]:
team_wins = team_matches.groupby('Team')['Win'].sum()

In [ ]:
team_wins

In [ ]:
team_draws = team_matches.groupby('Team')['Draw'].sum()

In [ ]:
team_draws

In [ ]:
team_losses = team_matches.groupby('Team')['Loss'].sum()

In [ ]:
team_losses

In [ ]:
league_table = pd.DataFrame({
    'Wins': team_wins,
    'Draws': team_draws,
    'Losses': team_losses
})

In [ ]:
league_table

In [ ]:
league_table['Played'] = (
    league_table['Wins']
    + league_table['Draws']
    + league_table['Losses']
)

In [ ]:
league_table

In [ ]:
team_goals = team_matches.groupby('Team')['Goals'].sum()

In [ ]:
team_goals

In [ ]:
team_goals_conceded = team_matches.groupby('Team')['GoalsConceded'].sum()

In [ ]:
team_goals_conceded

In [ ]:
team_goals.sum(), team_goals_conceded.sum()

In [ ]:
league_table['GoalsFor'] = team_goals
league_table['GoalsAgainst'] = team_goals_conceded

In [ ]:
league_table

In [ ]:
league_table['GoalDifference'] = (league_table['GoalsFor'] - league_table['GoalsAgainst'])

In [ ]:
league_table

In [ ]:
league_table['Points'] = (league_table['Wins'] * 3 + league_table['Draws'])

In [ ]:
league_table

In [ ]:
league_table = league_table.sort_values(['Points', 'GoalDifference', 'GoalsFor'], ascending=[False, False, False])

In [ ]:
league_table

In [ ]:
league_table['Position'] = range(1, len(league_table) + 1)

In [ ]:
league_table

In [ ]:
league_table = league_table[['Position', 'Played', 'Wins', 'Draws', 'Losses', 'GoalsFor', 'GoalsAgainst', 'GoalDifference', 'Points']]

In [ ]:
league_table

In [ ]:
(league_table['Wins'] + league_table['Losses'] + league_table['Draws'] == league_table['Played']).all()

In [ ]:
(
    league_table['Wins'] * 3
    + league_table['Draws']
    == league_table['Points']
).all()

In [ ]:
(
    league_table['GoalsFor']
    - league_table['GoalsAgainst']
    == league_table['GoalDifference']
).all()

In [ ]:
league_table.to_csv('../data/processed/league_table.csv', index=False)

In [ ]:
pd.read_csv('../data/processed/league_table.csv').head()

In [ ]:
league_table.columns

In [ ]:
league_table = league_table.reset_index()

In [ ]:
league_table.columns

In [ ]:
league_table = league_table[
    [
        'Position',
        'Team',
        'Played',
        'Wins',
        'Draws',
        'Losses',
        'GoalsFor',
        'GoalsAgainst',
        'GoalDifference',
        'Points'
    ]
]

In [ ]:
league_table.head()

In [ ]:
league_table.to_csv(
    '../data/processed/league_table.csv',
    index=False
)

In [ ]:
saved_league_table = pd.read_csv(
    '../data/processed/league_table.csv'
)

saved_league_table.columns

In [ ]:
saved_league_table.equals(league_table)

In [ ]:
team_matches.to_csv(
    '../data/processed/team_matches.csv',
    index=False
)

In [ ]:
pd.read_csv(
    '../data/processed/team_matches.csv'
).shape

In [ ]:
team_matches.head()

In [ ]:
team_matches.groupby('Team')['Goals'].mean().sort_values(ascending=False)

In [ ]:
team_matches.groupby('Team')['GoalsConceded'].mean().sort_values()

In [ ]:
team_matches.groupby('Team').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean')
)

In [ ]:
team_metrics = team_matches.groupby('Team').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean')
)

In [ ]:
team_metrics['GoalDifferencePerMatch'] = (
    team_metrics['GoalsPerMatch']
    - team_metrics['GoalsConcededPerMatch']
)

In [ ]:
team_metrics.sort_values(
    'GoalDifferencePerMatch',
    ascending=False
)

In [ ]:
team_metrics.index

In [ ]:
league_table.index

In [ ]:
team_metrics = team_metrics.reset_index()

In [ ]:
team_metrics.head()

In [ ]:
team_performance = league_table.merge(
    team_metrics,
    on='Team',
    how='left'
)

In [ ]:
team_performance.head()

In [ ]:
team_performance.isna().sum()

In [ ]:
team_performance.to_csv(
    '../data/processed/team_performance.csv',
    index=False
)

In [ ]:
team_shot_metrics = team_matches.groupby('Team').agg(
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean')
)

team_shot_metrics.head()

In [ ]:
home_away_goals = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean')
)

home_away_goals.head(10)

In [ ]:
home_away_goals = home_away_goals.reset_index()

home_away_goals.head()

In [ ]:
home_away_comparison = home_away_goals.pivot(
    index='Team',
    columns='Venue',
    values=['GoalsPerMatch', 'GoalsConcededPerMatch']
)

home_away_comparison.head()

In [ ]:
home_away_comparison.columns = [
    'AwayGoalsPerMatch',
    'HomeGoalsPerMatch',
    'AwayGoalsConcededPerMatch',
    'HomeGoalsConcededPerMatch'
]

home_away_comparison = home_away_comparison.reset_index()

home_away_comparison.head()

In [ ]:
home_away_comparison['HomeGoalAdvantage'] = (
    home_away_comparison['HomeGoalsPerMatch']
    - home_away_comparison['AwayGoalsPerMatch']
)

home_away_comparison['HomeDefensiveAdvantage'] = (
    home_away_comparison['AwayGoalsConcededPerMatch']
    - home_away_comparison['HomeGoalsConcededPerMatch']
)

home_away_comparison.head()

In [ ]:
home_away_comparison.sort_values(
    'HomeGoalAdvantage',
    ascending=False
)[
    [
        'Team',
        'AwayGoalsPerMatch',
        'HomeGoalsPerMatch',
        'HomeGoalAdvantage'
    ]
]

In [ ]:
home_away_comparison.sort_values(
    'HomeDefensiveAdvantage',
    ascending=False
)[
    [
        'Team',
        'AwayGoalsConcededPerMatch',
        'HomeGoalsConcededPerMatch',
        'HomeDefensiveAdvantage'
    ]
]

In [ ]:
home_away_comparison['HomePerformanceDifference'] = (
    home_away_comparison['HomeGoalAdvantage']
    + home_away_comparison['HomeDefensiveAdvantage']
)

home_away_comparison.sort_values(
    'HomePerformanceDifference',
    ascending=False
)[
    [
        'Team',
        'HomeGoalAdvantage',
        'HomeDefensiveAdvantage',
        'HomePerformanceDifference'
    ]
]

In [ ]:
home_away_points = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    PointsPerMatch=('Points', 'mean')
)

home_away_points = home_away_points.reset_index()

home_away_points.head(10)

In [ ]:
home_away_points_comparison = home_away_points.pivot(
    index='Team',
    columns='Venue',
    values='PointsPerMatch'
)

home_away_points_comparison.columns = [
    'AwayPointsPerMatch',
    'HomePointsPerMatch'
]

home_away_points_comparison = home_away_points_comparison.reset_index()

home_away_points_comparison['HomePointsAdvantage'] = (
    home_away_points_comparison['HomePointsPerMatch']
    - home_away_points_comparison['AwayPointsPerMatch']
)

home_away_points_comparison.sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
team_matches.columns.tolist()

In [ ]:
saved_team_matches = pd.read_csv(
    '../data/processed/team_matches.csv'
)

saved_team_matches.columns.tolist()

In [ ]:
home_away_points_comparison.sort_values(
    'HomePointsAdvantage',
    ascending=False
)[
    [
        'Team',
        'AwayPointsPerMatch',
        'HomePointsPerMatch',
        'HomePointsAdvantage'
    ]
]

In [ ]:
home_away_analysis = home_away_comparison.merge(
    home_away_points_comparison,
    on='Team',
    how='inner'
)

home_away_analysis[
    [
        'Team',
        'HomeGoalAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis[
    [
        'HomeGoalAdvantage',
        'HomePointsAdvantage'
    ]
].corr()

In [ ]:
home_away_shots = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean')
)

home_away_shots = home_away_shots.reset_index()
home_away_shots.head(10)

In [ ]:
home_away_shots_comparison = home_away_shots.pivot(
    index='Team',
    columns='Venue',
    values=[
        'ShotsPerMatch',
        'ShotsOnTargetPerMatch'
    ]
)

home_away_shots_comparison

In [ ]:
home_away_shots.columns.tolist()

In [ ]:
home_away_shots_comparison.columns = [
    'AwayShotsPerMatch',
    'HomeShotsPerMatch',
    'AwayShotsOnTargetPerMatch',
    'HomeShotsOnTargetPerMatch'
]

home_away_shots_comparison = home_away_shots_comparison.reset_index()

home_away_shots_comparison.head()

In [ ]:
home_away_shots_comparison['HomeShotsAdvantage'] = (
    home_away_shots_comparison['HomeShotsPerMatch']
    - home_away_shots_comparison['AwayShotsPerMatch']
)

home_away_shots_comparison['HomeShotsOnTargetAdvantage'] = (
    home_away_shots_comparison['HomeShotsOnTargetPerMatch']
    - home_away_shots_comparison['AwayShotsOnTargetPerMatch']
)

home_away_shots_comparison[
    [
        'Team',
        'HomeShotsAdvantage',
        'HomeShotsOnTargetAdvantage'
    ]
].sort_values(
    'HomeShotsAdvantage',
    ascending=False
)

In [ ]:
home_away_shots_comparison['AwayShotAccuracy'] = (
    home_away_shots_comparison['AwayShotsOnTargetPerMatch']
    / home_away_shots_comparison['AwayShotsPerMatch']
)

home_away_shots_comparison['HomeShotAccuracy'] = (
    home_away_shots_comparison['HomeShotsOnTargetPerMatch']
    / home_away_shots_comparison['HomeShotsPerMatch']
)

home_away_shots_comparison['HomeShotAccuracyAdvantage'] = (
    home_away_shots_comparison['HomeShotAccuracy']
    - home_away_shots_comparison['AwayShotAccuracy']
)

In [ ]:
home_away_shots_comparison[
    [
        'Team',
        'AwayShotAccuracy',
        'HomeShotAccuracy',
        'HomeShotAccuracyAdvantage'
    ]
].sort_values(
    'HomeShotAccuracyAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis = home_away_analysis.merge(
    home_away_shots_comparison[
        [
            'Team',
            'HomeShotsAdvantage',
            'HomeShotsOnTargetAdvantage',
            'HomeShotAccuracyAdvantage'
        ]
    ],
    on='Team',
    how='inner'
)

In [ ]:
home_away_analysis[
    [
        'HomeGoalAdvantage',
        'HomeShotsAdvantage',
        'HomeShotsOnTargetAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].corr()['HomePointsAdvantage'].sort_values(ascending=False)

In [ ]:
home_away_analysis.shape

In [ ]:
home_away_analysis.isna().sum()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    home_away_analysis['HomeShotAccuracyAdvantage'],
    home_away_analysis['HomePointsAdvantage']
)

plt.xlabel('Home Shot Accuracy Advantage')
plt.ylabel('Home Points Advantage')
plt.title('Home Shot Accuracy vs Home Points Advantage')

plt.axhline(0)
plt.axvline(0)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    home_away_analysis['HomeShotAccuracyAdvantage'],
    home_away_analysis['HomePointsAdvantage']
)

for _, row in home_away_analysis.iterrows():
    plt.annotate(
        row['Team'],
        (
            row['HomeShotAccuracyAdvantage'],
            row['HomePointsAdvantage']
        ),
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.xlabel('Home Shot Accuracy Advantage')
plt.ylabel('Home Points Advantage')
plt.title('Home Shot Accuracy vs Home Points Advantage')

plt.axhline(0)
plt.axvline(0)

plt.show()

In [ ]:
home_away_analysis[
    (home_away_analysis['HomeShotAccuracyAdvantage'] > 0) &
    (home_away_analysis['HomePointsAdvantage'] > 0)
][
    [
        'Team',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis[
    (home_away_analysis['HomeShotAccuracyAdvantage'] < 0) &
    (home_away_analysis['HomePointsAdvantage'] > 0)
][
    [
        'Team',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_analysis[
    [
        'Team',
        'HomeShotsAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomePointsAdvantage'
    ]
].sort_values(
    'HomePointsAdvantage',
    ascending=False
)

In [ ]:
home_away_shots_conceded = team_matches.groupby(
    ['Team', 'Venue']
).agg(
    ShotsConcededPerMatch=('ShotsConceded', 'mean'),
    ShotsOnTargetConcededPerMatch=('ShotsOnTargetConceded', 'mean')
)

home_away_shots_conceded = home_away_shots_conceded.reset_index()

In [ ]:
home_away_shots_conceded.head(10)

In [ ]:
home_away_shots_conceded_comparison = home_away_shots_conceded.pivot(
    index='Team',
    columns='Venue',
    values=[
        'ShotsConcededPerMatch',
        'ShotsOnTargetConcededPerMatch'
    ]
)

home_away_shots_conceded_comparison

In [ ]:
home_away_shots_conceded_comparison.columns = [
    'AwayShotsConcededPerMatch',
    'HomeShotsConcededPerMatch',
    'AwayShotsOnTargetConcededPerMatch',
    'HomeShotsOnTargetConcededPerMatch'
]

home_away_shots_conceded_comparison = (
    home_away_shots_conceded_comparison
    .reset_index()
)

home_away_shots_conceded_comparison.head()

In [ ]:
home_away_shots_conceded_comparison['HomeShotsPreventionAdvantage'] = (
    home_away_shots_conceded_comparison['AwayShotsConcededPerMatch']
    - home_away_shots_conceded_comparison['HomeShotsConcededPerMatch']
)

home_away_shots_conceded_comparison['HomeShotsOnTargetPreventionAdvantage'] = (
    home_away_shots_conceded_comparison['AwayShotsOnTargetConcededPerMatch']
    - home_away_shots_conceded_comparison['HomeShotsOnTargetConcededPerMatch']
)

In [ ]:
home_away_shots_conceded_comparison.head()

In [ ]:
home_away_analysis = home_away_analysis.merge(
    home_away_shots_conceded_comparison[
        [
            'Team',
            'HomeShotsPreventionAdvantage',
            'HomeShotsOnTargetPreventionAdvantage'
        ]
    ],
    on='Team',
    how='inner'
)

In [ ]:
home_away_analysis.shape

In [ ]:
home_away_analysis[
    [
        'HomeShotsAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomeShotsOnTargetAdvantage',
        'HomeShotsPreventionAdvantage',
        'HomeShotsOnTargetPreventionAdvantage',
        'HomePointsAdvantage'
    ]
].corr()['HomePointsAdvantage'].sort_values(ascending=False)

In [ ]:
correlations = home_away_analysis[
    [
        'HomeShotsAdvantage',
        'HomeShotAccuracyAdvantage',
        'HomeShotsOnTargetAdvantage',
        'HomeShotsPreventionAdvantage',
        'HomeShotsOnTargetPreventionAdvantage'
    ]
].corrwith(
    home_away_analysis['HomePointsAdvantage']
).sort_values(ascending=False)

correlations

In [ ]:
plt.figure(figsize=(10, 6))

correlations.sort_values().plot(kind='barh')

plt.xlabel('Correlation with Home Points Advantage')
plt.ylabel('Metric')
plt.title('Home Performance Metrics vs Home Points Advantage')
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

correlations.sort_values().plot(kind='barh')

plt.xlabel('Correlation with Home Points Advantage')
plt.ylabel('Metric')
plt.title('Home Performance Metrics vs Home Points Advantage')
plt.xlim(-0.6, 0.6)
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
team_summary = team_matches.groupby('Team').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean'),
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean'),
    ShotsConcededPerMatch=('ShotsConceded', 'mean'),
    ShotsOnTargetConcededPerMatch=('ShotsOnTargetConceded', 'mean'),
    PointsPerMatch=('Points', 'mean')
).reset_index()

team_summary

In [ ]:
team_summary['ShotAccuracy'] = (
    team_summary['ShotsOnTargetPerMatch']
    / team_summary['ShotsPerMatch']
)

team_summary['OpponentShotAccuracy'] = (
    team_summary['ShotsOnTargetConcededPerMatch']
    / team_summary['ShotsConcededPerMatch']
)

team_summary

In [ ]:
team_correlations = team_summary[
    [
        'GoalsPerMatch',
        'GoalsConcededPerMatch',
        'ShotsPerMatch',
        'ShotsOnTargetPerMatch',
        'ShotsConcededPerMatch',
        'ShotsOnTargetConcededPerMatch',
        'ShotAccuracy',
        'OpponentShotAccuracy'
    ]
].corrwith(
    team_summary['PointsPerMatch']
).sort_values(ascending=False)

team_correlations

In [ ]:
plt.figure(figsize=(10, 6))

team_correlations.sort_values().plot(kind='barh')

plt.xlabel('Correlation with Points Per Match')
plt.ylabel('Metric')
plt.title('Team Performance Metrics vs Points Per Match')
plt.xlim(-1, 1)
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
result_comparison = team_matches.groupby('Result').agg(
    GoalsPerMatch=('Goals', 'mean'),
    GoalsConcededPerMatch=('GoalsConceded', 'mean'),
    ShotsPerMatch=('Shots', 'mean'),
    ShotsOnTargetPerMatch=('ShotsOnTarget', 'mean'),
    ShotsConcededPerMatch=('ShotsConceded', 'mean'),
    ShotsOnTargetConcededPerMatch=('ShotsOnTargetConceded', 'mean'),
    PointsPerMatch=('Points', 'mean')
).reindex(['W', 'D', 'L'])

result_comparison

In [ ]:
result_comparison['ShotAccuracy'] = (
    result_comparison['ShotsOnTargetPerMatch']
    / result_comparison['ShotsPerMatch']
)

result_comparison[['ShotAccuracy']]

In [ ]:
result_comparison['GoalConversion'] = (
    result_comparison['GoalsPerMatch']
    / result_comparison['ShotsPerMatch']
)

result_comparison[['GoalsPerMatch', 'ShotsPerMatch', 'GoalConversion']]

In [ ]:
result_comparison['GoalsPerShotOnTarget'] = (
    result_comparison['GoalsPerMatch']
    / result_comparison['ShotsOnTargetPerMatch']
)

result_comparison[['GoalsPerMatch', 'ShotsOnTargetPerMatch', 'GoalsPerShotOnTarget']]

In [ ]:
team_summary['GoalConversion'] = (
    team_summary['GoalsPerMatch']
    / team_summary['ShotsPerMatch']
)

team_summary[
    ['Team', 'GoalsPerMatch', 'ShotsPerMatch', 'GoalConversion', 'PointsPerMatch']
].sort_values('GoalConversion', ascending=False)

In [ ]:
team_summary[['GoalConversion', 'PointsPerMatch']].corr()

In [ ]:
defensive_correlations = team_summary[
    [
        'GoalsConcededPerMatch',
        'ShotsConcededPerMatch',
        'ShotsOnTargetConcededPerMatch',
        'OpponentShotAccuracy'
    ]
].corrwith(
    team_summary['PointsPerMatch']
).sort_values()

defensive_correlations

In [ ]:
team_performance[
    [
        'Position',
        'Team',
        'Points',
        'GoalsFor',
        'GoalsAgainst',
        'GoalDifference',
        'GoalsPerMatch',
        'GoalsConcededPerMatch',
        'PointsPerMatch'
    ]
]

In [ ]:
team_performance.columns.tolist()

In [ ]:
team_performance['PointsPerMatch'] = (
    team_performance['Points']
    / team_performance['Played']
)

In [ ]:
team_performance[
    [
        'Position',
        'Team',
        'Points',
        'PointsPerMatch',
        'GoalsFor',
        'GoalsAgainst',
        'GoalDifference',
        'GoalsPerMatch',
        'GoalsConcededPerMatch'
    ]
]

In [ ]:
%whos

## Project Data Dictionary

### Core datasets

| Variable | Grain | Description |
|---|---|---|
| `df` | 1 row = match | Raw Premier League match dataset imported from Football-Data.co.uk |
| `team_matches` | 1 row = team-match | Match data transformed into a team perspective, giving each team one row per match |
| `league_table` | 1 row = team | Final league standings calculated from match results |
| `team_performance` | 1 row = team | League standings combined with goals-per-match and goals-conceded-per-match metrics |
| `team_summary` | 1 row = team | Team attacking, defensive and points-per-match metrics |
| `home_away_analysis` | 1 row = team | Comparison of each team's home and away performance |
| `result_comparison` | 1 row = result type | Comparison of match statistics across wins, draws and losses |

### Supporting / intermediate datasets

| Variable | Purpose |
|---|---|
| `home` | Home-team perspective before combining with away matches |
| `away` | Away-team perspective before combining with home matches |
| `home_away_goals` | Intermediate home/away goal metrics |
| `home_away_comparison` | Intermediate home/away goal comparison |
| `home_away_points` | Intermediate home/away points-per-match calculations |
| `home_away_points_comparison` | Home vs away points comparison |
| `home_away_shots` | Intermediate home/away shooting metrics |
| `home_away_shots_comparison` | Home vs away shooting comparison |
| `home_away_shots_conceded` | Intermediate defensive shooting metrics |
| `home_away_shots_conceded_comparison` | Home vs away defensive shooting comparison |

### Statistical outputs

| Variable | Description |
|---|---|
| `correlations` | Correlations between home-performance metrics and home points advantage |
| `team_correlations` | Correlations between team metrics and points per match |
| `defensive_correlations` | Correlations between defensive metrics and points per match |

### Helper objects

| Variable | Purpose |
|---|---|
| `football_cols` | List of core football-performance columns |
| `get_result` | Function used to convert match outcomes into W/D/L from each team's perspective |

In [17]:
league_table

NameError: name 'league_table' is not defined

In [18]:
team_matches.groupby(['Team', 'Result']).size().loc[
    [['Man City', 'Man United']]
]

NameError: name 'team_matches' is not defined